# Dark.Tyro — stripping PhotoGuard off the text-to-image editing pipeline

**FIT5230 Malicious AI · Theme 2 (Text-to-Image) · Dark / attack side**
**Echo Zhao · Nissa Colidea** — Milestone 1, 28 August 2026

The IMPRESS baseline (NeurIPS 2023), repaired to run in 2026, end to end on one free GPU:
`clean → PhotoGuard-protect → IMPRESS-purify → edit → measure`.

**Run it on Kaggle, Accelerator = GPU T4 x2.** Not the P100 — it is `sm_60`, and current
PyTorch wheels ship no kernels for it (repair 11). Run top to bottom; each configuration
zips itself the moment it finishes, so download the zip then.

## Eleven repairs to the reference implementation

The published code does not run as-is. 1–6 are what it takes to get one successful run;
7–9 are bookkeeping faults that make *multi-configuration* results silently wrong; 10–11 are
hardware limits. Each is marked `FIX n` in the code.

| # | fault | where |
|---|---|---|
| 1 | `runwayml/stable-diffusion-inpainting` deleted from Hugging Face | cell 3 |
| 2 | mirror has no `revision="fp16"` branch | cell 3 |
| 3 | `sewar` imported by `pg_metric.py`, absent from `requirements.txt` | cell 3 |
| 4 | launcher shards across 4 GPUs | cell 6 (`--parallel_index=-1`) |
| 5 | protect writes `adv_*`, purify reads `adapt_adv_*` — no bridge | cell 6 |
| 6 | `pg_generate.py` defaults `--diff_steps=50` while protect used `4`; folder names are built from params, so the chain silently reads an empty directory | cell 6 |
| 7 | **the purified folder name encodes no `pg_*` parameters** — every configuration overwrites the last | cell 6 (wipe + stamped archive) |
| 8 | bridge step used `glob(...)[0]`, an arbitrary pick once several runs exist | cell 6 (paths rebuilt from params) |
| 9 | display cell hard-coded one configuration's parameters | cell 5 (single `PARAMS` dict) |
| 10 | `(200, 10)` exhausts 16 GB of GPU memory | cell 5 (allocator config; that point omitted by design) |
| 11 | Kaggle P100 is `sm_60`; current torch wheels start at `sm_70` | cell 1 (gate) |

Fixes 7 and 8 are the dangerous ones: they do not raise an error, they hand you the previous
run's images under this run's label.

In [ ]:
# 1 · Platform, device, paths, persistence. Everything below derives from this cell.
import os, sys, re, pathlib, subprocess, torch

BASE, PLATFORM = ((pathlib.Path('/kaggle/working'), 'kaggle') if os.path.isdir('/kaggle/working')
                  else (pathlib.Path('/content'), 'colab') if os.path.isdir('/content')
                  else (pathlib.Path.cwd(), 'local'))

# fp16 backward is unreliable off CUDA, so MPS/CPU run fp32 (see the patch in cell 3).
DEVICE, USE_FP32 = (('cuda:0', False) if torch.cuda.is_available()
                    else ('mps', True) if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available()
                    else ('cpu', True))

IMPRESS_DIR = BASE / 'Impress'
ROOT        = BASE / 'helen_face'     # must stay a SIBLING of Impress/: the scripts use ../helen_face
ARCHIVE     = BASE / 'tyro_results'

SUBENV = dict(os.environ,
              PYTORCH_ENABLE_MPS_FALLBACK='1',
              PYTORCH_MPS_HIGH_WATERMARK_RATIO='0.0',
              # FIX 10: 2000 backwards through an unrolled UNet fragment the CUDA allocator;
              # it fails with "tried to allocate 2 GiB" while memory is still free.
              PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')

# FIX 11 · a torch wheel only contains kernels for the architectures it was built for.
# On an unsupported card everything loads, then the first real op dies two minutes in with
# "no kernel image is available for execution on the device". Catch it here, not there.
GPU_OK = True
if DEVICE.startswith('cuda'):
    _tag  = 'sm_%d%d' % torch.cuda.get_device_capability(0)
    GPU_OK = _tag in torch.cuda.get_arch_list()
    print(f'gpu    : {torch.cuda.get_device_name(0)} ({_tag}) · torch {torch.__version__}')
    if not GPU_OK:
        print(f'!! this torch build has no kernels for {_tag}. Switch accelerator to GPU T4 x2.')

# Persistence. Kaggle keeps /kaggle/working only if you Save Version or download the zip;
# Colab wipes the VM on disconnect, so results go to Drive.
if PLATFORM == 'colab':
    try:
        from google.colab import drive; drive.mount('/content/drive')
        ARCHIVE = pathlib.Path('/content/drive/MyDrive/FIT5230_sweep')
    except Exception as e:
        print(f'!! Drive mount failed ({e}) — download tyro_results/ before disconnecting.')
ARCHIVE.mkdir(parents=True, exist_ok=True)
print(f'{PLATFORM} · {DEVICE} · {"fp32" if USE_FP32 else "fp16"} · results -> {ARCHIVE}')

In [ ]:
# 2 · Clone IMPRESS, install only what is genuinely missing, patch the 2023 source.
#
# Do NOT `pip install -r Impress/requirements.txt` here. It lists torch, torchvision,
# accelerate, datasets and tensorboard with no version pins; pip then fetches a newer torch
# plus the whole nvidia-* stack — several GB, ~20 min, and often a CUDA mismatch afterwards.
# Kaggle and Colab already ship those. Install the small pure-Python gaps only.
import importlib.util

# sewar: FIX 3, imported by pg_metric.py but missing from requirements.txt.
# gdown: ours — the dataset is a Google Drive link in the README, not a pip package.
# ftfy:  in requirements.txt; a transitive need of the CLIP tokenizer, imported by no script.
NEEDED = {'sewar':'sewar', 'lpips':'lpips', 'gdown':'gdown', 'ftfy':'ftfy', 'diffusers':'diffusers'}
missing = [pkg for mod, pkg in NEEDED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run(f'pip install -q --no-deps {" ".join(missing)}', shell=True, check=False)
print('installed:', missing or 'nothing needed')

if not IMPRESS_DIR.exists():
    subprocess.run(f'git clone -q --depth 1 https://github.com/AAAAAAsuka/Impress.git {IMPRESS_DIR}',
                   shell=True, check=True)

# Patched in Python, not sed: BSD sed on macOS rejects `sed -i 's/../../'`.
n = 0
for p in sorted(IMPRESS_DIR.glob('*.py')):
    s = orig = p.read_text()
    s = s.replace('runwayml/stable-diffusion-inpainting',                  # FIX 1: repo deleted
                  'stable-diffusion-v1-5/stable-diffusion-inpainting')
    s = re.sub(r'^[ \t]*revision="fp16",[ \t]*\r?\n', '', s, flags=re.M)   # FIX 2: no fp16 branch
    if USE_FP32:
        s = s.replace('torch.float16', 'torch.float32').replace('.half()', '.float()')
    if s != orig:
        p.write_text(s); n += 1
print(f'cloned + patched {n} files')

In [ ]:
# 3 · Hugging Face (non-blocking) and the Helen face data.
# SD 1.5 is public, so anonymous download normally works — this only picks up a token if one
# is already there. `notebook_login()` is deliberately not called: it blocks forever waiting
# for a paste, which looks exactly like a hung cell.
try:
    from huggingface_hub import login, whoami
    tok = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
    if not tok and PLATFORM == 'kaggle':
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret('HF_TOKEN')   # Add-ons -> Secrets -> HF_TOKEN
    if tok: login(token=tok, add_to_git_credential=False)
    print('hf user:', whoami().get('name'))
except Exception:
    print('hf: anonymous (fine — SD 1.5 is public; only revisit this on a 401)')

# Dataset: the Google Drive link from the IMPRESS README, unzipped as a SIBLING of Impress/.
import zipfile, gdown
N_IMAGES = 2      # >= 2, or pg_metric's torch.std() returns nan and the metric is unreportable
zip_path = BASE / 'helen_face_dataset.zip'
if not zip_path.exists():
    gdown.download(id='16xISe7M_DlSqM2Zf2lWI4JJXcdsDEPsl', output=str(zip_path), quiet=True)
if not (ROOT / 'clean').exists():
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as z: z.extractall(ROOT)

keep = set(sorted(os.listdir(ROOT / 'clean'))[:N_IMAGES])   # clean/ and mask/ must stay in sync
for sub in ('clean', 'mask'):
    for f in os.listdir(ROOT / sub):
        if f not in keep: os.remove(ROOT / sub / f)
print(f'{N_IMAGES} face(s):', sorted(keep))

## Why the grid is `pg_iters` × `pg_grad_reps`

From `pg_mask_diff_helen.py`:

```python
grad_normalized = grad.detach() / (grad_norm + 1e-10)   # unit-length direction
X_adv = X_adv - grad_normalized * actual_step_size      # step_size = 1 -> each step moves L2 = 1.0
d_x_norm = torch.renorm(d_x, p=2, dim=0, maxnorm=eps)   # projected back into the radius-16 ball
```

Every PGD step moves the image exactly `pg_step_size` = 1.0 in L2 and is then projected back
inside a ball of radius `pg_eps` = 16. **From the centre you hit the wall after ~16 steps.**
Everything after that walks *around* the surface, not further out.

> **The leash.** `pg_eps` is the length of the dog's leash; `pg_iters` is how long the dog
> runs. Once the leash is taut, running longer does not get the dog one inch further from the
> post — it only lets it find a better spot along the arc.

So the grid is not "more is better". `(40, 10)` and `(200, 2)` cost the **same compute**, and
comparing them asks something the paper does not: *at a fixed budget, is it better to take
more steps or to estimate each step better?*

| lever | what it changes | cost |
|---|---|---|
| `pg_eps` 16 → 32 | the size of the ball — the real ceiling on shield strength | free (`type=int`) |
| `pg_grad_reps` 2 → 10 | averages 10 stochastic gradients instead of 2 → a truer direction (EOT) | linear, ×5 |
| `diff_steps` 4 → 8 | how much of the diffusion process the attack unrolls | linear, and the memory driver |
| `pg_iters` 40 → 200 | steps taken *inside* the ball | linear, **and saturated** |

`(200, 10)` exhausts 16 GB and is omitted by design: it spends 5× the compute of `(40, 10)`
on the lever that had already run out. Dropping `diff_steps` to rescue that one point would
mean a grid where one cell used a different attack, which is not a grid.

**Trap:** `--attack_type=linf` is a no-op here — `super_linf` clamps to `X ± eps` on images in
`[-1, 1]`, so at `pg_eps=16` the constraint never binds. Stay on `l2`.

In [ ]:
# 4 · FIX 9 — the one place parameters live. Every path, command and label below is an
#     f-string off this dict, so no two cells can drift apart.
#     Values follow the reference run in Impress/scripts/new/pg_mask_diff_test.sh, except
#     pur_iters (see the note) — argparse defaults differ and are NOT what the paper ran.
PARAMS = dict(
    # --- PhotoGuard: the DEFENCE we attack. Fixed across the sweep. ---
    attack_type  = 'l2',      # linf never binds at eps=16 on [-1,1] images
    pg_eps       = 16,        # L2 budget = the ceiling on shield strength (int only)
    pg_step_size = 1,         # each PGD step moves L2 = exactly 1.0
    pg_eta       = 1,         # DDIM eta inside the attack; 1 = stochastic, which is why grad_reps matters
    diff_steps   = 4,         # steps the attack unrolls — the memory driver (FIX 10)
    guidance     = 7.5,
    seed         = 0,         # fixed -> the 3-panel comparison is controlled

    # --- IMPRESS: OUR attack. Fixed, so pg_* is the only variable. ---
    pur_eps   = 0.1,          # free LPIPS allowance before the stay-close penalty switches on
    pur_iters = 100,          # NOTE: the reference script uses 1000. 100 is the argparse default
                              # and is what M1 ran; raising it is the first M2 experiment.
    pur_lr    = 0.005,
    pur_alpha = 0.01,
    pur_noise = 0.05,

    # --- The evaluation edit. Fixed, or the comparison means nothing. ---
    prompt          = 'a person in an airplane',
    test_guidance   = 7.5,
    test_diff_steps = 50,
)

CONFIGS = [(40, 2), (200, 2)]     # cheapest end-to-end check + strongest shield we can run
# CONFIGS = [(40, 2)]             # plumbing check only, ~5 min
# CONFIGS = [(40, 2), (40, 10), (200, 2)]   # the wider grid; (200,10) omitted by design

# Rough ETA, calibrated on our own measured (40,2) = 4.9 min at N_IMAGES=2 on a P100. The
# attack is backward-heavy through an unrolled UNet, so it tracks memory BANDWIDTH, not FLOPs
# — a T4 (320 GB/s) is ~2.2x slower than a P100 (732 GB/s). Key the scale on the actual card.
_SCALE = {'P100':1.0, 'T4':2.2, 'V100':0.6, 'A100':0.35, 'L4':1.5, 'P4':3.0}
_card  = torch.cuda.get_device_name(0) if DEVICE.startswith('cuda') else DEVICE
_s     = next((v for k, v in _SCALE.items() if k in _card), 39.0 if DEVICE=='cpu' else 2.6 if DEVICE=='mps' else 1.5)
eta    = lambda i, g: (i * g * N_IMAGES * 0.71/60) * _s + 3.0

print(f'{len(CONFIGS)} config(s) x {N_IMAGES} images on {_card} (x{_s} vs our P100 baseline)')
for i, g in CONFIGS: print(f'  ({i:>3}, {g:>2})  ~{eta(i, g):5.1f} min')
print(f'  {"total":>9}  ~{sum(eta(i, g) for i, g in CONFIGS):5.1f} min')

## The harness

Paths are rebuilt exactly as the source builds them — including its quirk of using
`re.sub("pur", "pur_diff", ...)`, which rewrites **every** occurrence of `pur` in the string,
not just the first. Guessing that folder name by hand is how you read an empty directory and
conclude the attack failed.

Three properties the reference scripts do not have:

- **Images are archived before metrics are parsed.** The images are the deliverable; a
  metric-parsing failure must never discard a finished 15-minute run.
- **A heartbeat.** `subprocess.run(..., stdout=PIPE)` buffers until exit, so a working stage
  and a hung one look identical. A background thread prints elapsed time every 60 s.
- **Stale purified folders are wiped** before each configuration (FIX 7).

In [ ]:
# 5 · Path helpers, stage runner, metric parser, and the per-configuration chain.
import shutil, json, time, threading
P = PARAMS

# FIX 8: rebuild these from parameters — never glob for them.
def adv_dir(i, g):
    return (f"{ROOT}/adv_{P['attack_type']}_eps{P['pg_eps']}_step{P['pg_step_size']}"
            f"_iter{i}grad_reps{g}_eta{P['pg_eta']}_diff_steps{P['diff_steps']}"
            f"_guidance{P['guidance']}_seed{P['seed']}")
adapt_dir    = lambda i, g: adv_dir(i, g).replace('/adv_', '/adapt_adv_')
adv_diff_dir = lambda i, g: re.sub('adv', 'adv_diff', adv_dir(i, g))
pur_dir      = lambda: (f"{ROOT}/pur_eps{P['pur_eps']}_pur_iters{P['pur_iters']}_pur_lr{P['pur_lr']}"
                        f"_pur_alpha{P['pur_alpha']}_pur_noise{P['pur_noise']}/")
pur_diff_dir = lambda: re.sub('pur', 'pur_diff', pur_dir())    # mirrors the source's global re.sub
tag_of       = lambda i, g: f'iter{i}_grad{g}_eps{P["pg_eps"]}'

# FIX 4: --parallel_index=-1 disables the 4-GPU sharding the launcher scripts assume.
PG  = (f"--attack_type={P['attack_type']} --pg_eps={P['pg_eps']} --pg_step_size={P['pg_step_size']} "
       f"--pg_eta={P['pg_eta']} --parallel_index=-1 --device={DEVICE}")
PUR = (f"--pur_eps={P['pur_eps']} --pur_iters={P['pur_iters']} --pur_lr={P['pur_lr']} "
       f"--pur_alpha={P['pur_alpha']} --pur_noise={P['pur_noise']}")

def sh(cmd, heartbeat=60):
    print(f'$ {cmd}', flush=True)
    t0, stop = time.time(), threading.Event()
    def _beat():
        while not stop.wait(heartbeat):
            print(f'   ... {(time.time()-t0)/60:.1f} min elapsed', flush=True)
    threading.Thread(target=_beat, daemon=True).start()
    try:
        r = subprocess.run(f'{sys.executable} -u {cmd}', shell=True, cwd=str(IMPRESS_DIR),
                           env=SUBENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    finally:
        stop.set()
    print(r.stdout[-3000:], f'\n   [{(time.time()-t0)/60:.1f} min]', flush=True)
    return r.stdout

# NumPy 2 prints {'ssim': [np.float64(0.53), ...]}; ast.literal_eval rejects that because
# np.float64(...) parses as a CALL, not a literal. Kaggle ships NumPy 2, Colab shipped 1.x —
# which is why this worked in one place and not the other. Pull the numbers out with a regex.
_NUM  = r'(?:np\.\w+\(\s*)?(-?(?:\d+\.?\d*(?:[eE][-+]?\d+)?|nan|inf))\s*\)?'
_PAIR = re.compile(r"'(\w+)'\s*:\s*\[\s*" + _NUM + r'\s*,\s*' + _NUM + r'\s*\]')

def parse_metric(out):
    return {m.group(1): {k: [float(a), float(b)] for k, a, b in _PAIR.findall(m.group(2))}
            for m in (re.match(r'^(adv|pur):\s*(\{.*\})\s*$', l.strip()) for l in out.splitlines()) if m}

def run_config(iters, reps):
    tag = tag_of(iters, reps)
    print('=' * 70, f'\n  CONFIG {tag}  (expect ~{eta(iters, reps):.0f} min)\n', '=' * 70, flush=True)
    t0 = time.time()

    # FIX 7: the purified folder name carries no pg_* parameters, so every configuration
    # writes to the SAME directory, with no skip-if-exists guard and no error. A leftover
    # folder means this run silently reports the PREVIOUS configuration's purified images.
    for stale in (pur_dir(), pur_diff_dir()):
        if os.path.exists(stale): shutil.rmtree(stale)

    # (a) protect — PhotoGuard. Skips images already done, so a re-run resumes.
    sh(f'pg_mask_diff_helen.py {PG} --pg_iters={iters} --pg_grad_reps={reps} '
       f'--diff_steps={P["diff_steps"]} --guidance={P["guidance"]}')

    # (b) FIX 5: bridge the broken folder contract between protect and purify.
    src, dst = adv_dir(iters, reps), adapt_dir(iters, reps)
    assert os.path.isdir(src), f'protect stage produced nothing at {src}'
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree(src, dst)

    # (c) purify — IMPRESS, our baseline attack.
    sh(f'pg_mask_pur_helen.py {PG} --pg_iters={iters} --pg_grad_reps={reps} '
       f'--diff_steps={P["diff_steps"]} {PUR}')

    # (d) edit all three variants. FIX 6: diff_steps forced through, or the folder name is wrong.
    common = (f'{PG} --pg_iters={iters} --pg_grad_reps={reps} --diff_steps={P["diff_steps"]} '
              f'--guidance={P["guidance"]} {PUR} --test_guidance={P["test_guidance"]} '
              f'--test_diff_steps={P["test_diff_steps"]} --prompt="{P["prompt"]}"')
    sh(f'pg_generate.py {common}')

    # Archive FIRST, under a stamped tag — the other half of FIX 7: the run becomes an
    # artefact that records which purification belongs to which protection. Then zip, so a
    # finished configuration is downloadable immediately instead of after the whole sweep.
    dest = ARCHIVE / tag; dest.mkdir(parents=True, exist_ok=True)
    for name, path in [('clean', f'{ROOT}/clean'), ('protected', adv_dir(iters, reps)),
                       ('purified', pur_dir()), ('edit_clean', f'{ROOT}/clean_diff'),
                       ('edit_protected', adv_diff_dir(iters, reps)), ('edit_purified', pur_diff_dir())]:
        if os.path.isdir(path): shutil.copytree(path, dest / name, dirs_exist_ok=True)
    (dest / 'params.json').write_text(json.dumps({**P, 'pg_iters': iters, 'pg_grad_reps': reps}, indent=2))
    zp = shutil.make_archive(str(ARCHIVE / tag), 'zip', root_dir=str(dest))
    print(f'[saved] {zp} ({os.path.getsize(zp)/1e6:.1f} MB) <- DOWNLOADABLE NOW', flush=True)

    # Metrics last, and never fatal: the images are the deliverable, metrics can be recomputed.
    raw = sh(f'pg_metric.py {common}')
    (dest / 'metric_stdout.txt').write_text(raw)
    print(f'[done] {tag} ({(time.time()-t0)/60:.1f} min)', flush=True)
    return parse_metric(raw)

print('harness ready ·', pur_dir())

In [ ]:
# 6 · RUN. Archived after each configuration, so a disconnect costs only the one in flight.
assert GPU_OK, 'this torch build has no kernels for this GPU — switch to T4 and re-run cell 1'

RESULTS = {}
for iters, reps in CONFIGS:
    try:
        RESULTS[(iters, reps)] = run_config(iters, reps)
    except Exception as e:                      # one dead configuration must not cost the grid
        print(f'!! ({iters}, {reps}) failed: {type(e).__name__}: {e}')
        RESULTS[(iters, reps)] = None
        if DEVICE.startswith('cuda'): torch.cuda.empty_cache()

(ARCHIVE / 'scores.json').write_text(
    json.dumps({f'{i}_{g}': v for (i, g), v in RESULTS.items()}, indent=2))
print('\ndone ->', ARCHIVE / 'scores.json')

## Reading the numbers

⚠️ **`pg_metric.py` compares the *edited* images against the *edited clean* image — not
against the original photograph.** From the source:

```python
clean_image = load_image(diff_dir_clean, image_name)   # the EDITED clean image
image_quality_metrics(clean_image, adv_image, adv_score_dict)
image_quality_metrics(clean_image, pur_image, pur_score_dict)
```

So these numbers answer *"did the editing pipeline behave normally?"* — **not** *"is the photo
intact?"* Two different things are called SSIM in this project and conflating them is the
easiest mistake available here.

- **`adv`** — edited-protected vs edited-clean. **Lower = the shield worked better.**
- **`pur`** — edited-purified vs edited-clean. **Higher = our purification worked better.**
- **`R_pipe = (pur − adv) / (1 − adv)`** — the share of the shield's damage that purification
  undid. SSIM = 1.0 is by construction "the pipeline behaved as if the photo had never been
  protected", which gives a principled denominator with no third measurement. It is a
  *pipeline* rate, **not** an edit-success rate: it says nothing about whether the output
  matches the prompt. That second measure is M2 work.

In [ ]:
# 7 · Results, R_pipe, and the sign test.
import pandas as pd

rows = [dict(config=f'{i}:{g}', units=i*g, metric=m,
             adv=round(sc['adv'][m][0], 4), pur=round(sc['pur'][m][0], 4),
             recovery=round(sc['pur'][m][0] - sc['adv'][m][0], 4))
        for (i, g), sc in RESULTS.items() if sc
        for m in ('ssim', 'psnr', 'vifp')
        if m in sc.get('adv', {}) and m in sc.get('pur', {})]
df = pd.DataFrame(rows).sort_values(['metric', 'units']) if rows else pd.DataFrame()
display(df)

if len(df):
    ss = df[df.metric == 'ssim'].copy()
    ss['R_pipe'] = ((ss.pur - ss.adv) / (1 - ss.adv) * 100).round(2)
    print('\nPIPELINE RESTORATION RATE  R_pipe = (pur - adv) / (1 - adv),  %\n')
    display(ss[['config', 'units', 'adv', 'pur', 'recovery', 'R_pipe']])

    # With two images per configuration any single metric can wander. But psnr, ssim and vifp
    # are three different instruments on the same pair of pictures: if all three agree on the
    # SIGN, something real moved. If they disagree, the recovery is indistinguishable from
    # zero — and saying so is a result, not a failure to get one.
    print('\nDirection agreement across the three metrics:')
    for cfg in df.config.unique():
        sub  = df[df.config == cfg]
        one  = len({r.recovery > 0 for _, r in sub.iterrows()}) == 1
        print(f'   {cfg:>8}  ' + '  '.join(f'{r.metric} {"+" if r.recovery > 0 else "-"}' for _, r in sub.iterrows())
              + ('   -> all three agree: real movement' if one else '   -> DISAGREE: within noise of zero'))

In [ ]:
# 8 · The headline figure, then a check that the results are somewhere durable.
#     Same seed, mask and prompt in every panel — pg_generate.py re-seeds numpy and torch
#     immediately before each generation — so every visible difference comes from the INPUT
#     IMAGE ALONE. That is what makes this a controlled comparison.
import matplotlib.pyplot as plt, glob
from PIL import Image

done = [c for c in CONFIGS if (ARCHIVE / tag_of(*c) / 'edit_clean').is_dir()]
assert done, 'nothing archived yet — run cell 6 first'
name = sorted(os.listdir(ARCHIVE / tag_of(*done[0]) / 'edit_clean'))[0]

fig, axes = plt.subplots(len(done), 3, figsize=(12, 4*len(done)), squeeze=False)
for r, (i, g) in enumerate(done):
    for c, (title, sub) in enumerate([('edited CLEAN', 'edit_clean'),
                                      ('edited PROTECTED', 'edit_protected'),
                                      ('edited PURIFIED', 'edit_purified')]):
        f = ARCHIVE / tag_of(i, g) / sub / name
        if f.exists(): axes[r][c].imshow(Image.open(f))
        axes[r][c].set_xticks([]); axes[r][c].set_yticks([])
        if r == 0: axes[r][c].set_title(title, fontsize=13)
    axes[r][0].set_ylabel(f'iters={i}\ngrad_reps={g}', fontsize=11)
fig.suptitle(f'Shield-strength sweep · "{P["prompt"]}" · seed {P["seed"]} (identical in every panel)', fontsize=14)
fig.tight_layout(); fig.savefig(ARCHIVE / 'sweep_grid.png', dpi=110, bbox_inches='tight'); plt.show()

zips = sorted(glob.glob(str(ARCHIVE / '*.zip')))
print(f'\n{len(zips)} zip(s) in {ARCHIVE}:', [f'{os.path.basename(z)} ({os.path.getsize(z)/1e6:.0f} MB)' for z in zips])
if PLATFORM == 'kaggle':
    print('NOT YET SAFE — download them from the Output panel, or Save Version (Commit), before closing.')

---

# Limitations

Stated plainly, because a claim we cannot evidence costs more than a modest result.

1. **Scale.** Two faces per configuration — enough for `pg_metric` to report a real standard
   deviation instead of `nan`, and enough for a controlled visual comparison, but not enough
   to rank two configurations against each other. We do not attempt that ranking.
2. **How to read it at n = 2.** Never trust one metric on two images; trust whether PSNR,
   SSIM and VIF agree on the **sign**. Agreement means something moved, disagreement means
   it is inside the noise — and saying so is a result.
3. **Run-to-run spread.** The RNG is seeded, but fp16 kernel selection and accumulation order
   are not fixed across GPU model and torch build. Repeat runs of identical parameters differ
   by about ±5 percentage points of `R_pipe`. At n = 2 the noise is the size of the effect.
4. **`pg_eps` is untested.** We vary `pg_iters` and `pg_grad_reps` only. Both 400-unit points
   landing near zero is *consistent* with `pg_iters` saturating, but equally consistent with
   both shields simply winning. This experiment cannot separate those, and we do not claim it.
5. **`pur_iters = 100`, where the reference script uses 1000.** Our own attack ran at a tenth
   of the paper's optimisation budget. That is a candidate explanation for the small recovery
   and the first thing M2 tests.
6. **`pg_metric`'s SSIM compares edited images to the edited clean image**, not to the original
   photograph. `R_pipe` is built on that question only.
7. **Why the recovery is small is an open question**, and we treat it as one. The diagnostics
   are deliberately outside this notebook; they are M2 work and we quote no number from them.

# What M2 adds

- **`pur_iters` 100 → 1000** and `pur_eps` 0.1 → 0.3: does our purifier move further than the
- ≥ 5 faces (the cheapest way to shrink the error bar), and a **`pg_eps` sweep 16 → 24 → 32** —
  the lever that raises the shield's ceiling, which `pg_iters` does not.
  VAE round-trip floor once it is given the paper's budget?
- A quantitative **edit-success** rate (CLIPScore against the prompt) beside `R_pipe`, and
  **LPIPS** beside SSIM.
- The cheap-purifier ablation and the frequency-targeted filter, on one fidelity-vs-success chart.